In [ ]:
import duckdb
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


# run utils.db_setup to set up duckdb database
conn = duckdb.connect('amazon.db')
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.displaycon=False
%config SqlMagic.autopandas=True

## Schema
![](https://relbench.stanford.edu/img/amazon-reviews.png)

## Tasks

### `rel-amazon-user-ltv`

Task is to predict the sum of prices of the products purchased and reviewed by a user in the next 2 years.

Filter: Consider only users who wrote a review in the past two years before timestamp.

Regression

Metric: MAE

In [ ]:
%%sql train_df <<
from user_ltv_train;

In [ ]:
train_df.head()

In [ ]:
print('TRAIN\n')

print(f'{len(train_df):,} samples over {train_df.customer_id.nunique():,} customers.')
print()
print('Temporal dist of samples:')
print(train_df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print('Temporal dist of customers (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    train_df
    .drop_duplicates(subset='customer_id', keep='last')
    .groupby('timestamp').customer_id.nunique() / train_df.customer_id.nunique()
)
print()
print('Label Distribution:')
mean = train_df.ltv.mean()
std = train_df.ltv.std()
print(f'Mean (std): {mean:,.1f} ± {std:,.1f}')
print(f'Prop zeros: {(train_df.ltv == 0).mean():.1%}')
print(f'Quantiles: \n{train_df.ltv.quantile(np.arange(0, 1.1, 0.1))}')

In [ ]:
%%sql val_df <<
from user_ltv_val;

In [ ]:
print('VAL\n')

print(f'{len(val_df):,} samples over {val_df.customer_id.nunique():,} customers.')
print()
print('Temporal dist of samples:')
print(val_df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print('Temporal dist of customers (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    val_df
    .drop_duplicates(subset='customer_id', keep='last')
    .groupby('timestamp').customer_id.nunique() / val_df.customer_id.nunique()
)
print()
print(f'Customers in val seen in train: {len(set(val_df.customer_id) & set(train_df.customer_id)):,}')
print()
print('Label Distribution:')
mean = val_df.ltv.mean()
std = val_df.ltv.std()
print(f'Mean (std): {mean:,.1f} ± {std:,.1f}')
print(f'Prop zeros: {(val_df.ltv == 0).mean():.1%}')
print(f'Quantiles: \n{val_df.ltv.quantile(np.arange(0, 1.1, 0.1))}')

### `rel-amazon-user-churn`

Predict if the user will review a product in the next 2 years.

Filter: Consider only users who wrote a review in the past two years.

Classification

Metric: AP

In [ ]:
%%sql train_df <<
from user_churn_train;

In [ ]:
train_df.head()

In [ ]:
print('TRAIN\n')

print(f'{len(train_df):,} samples over {train_df.customer_id.nunique():,} users.')
print()
print('Label Distribution:')
print(train_df.churn.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
print(train_df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print('Temporal dist of users (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    train_df
    .drop_duplicates(subset='customer_id', keep='last')
    .groupby('timestamp').customer_id.nunique() / train_df.customer_id.nunique()
)

In [ ]:
%%sql val_df <<
from user_churn_val;

In [ ]:
print('VAL\n')

print(f'{len(val_df):,} samples over {val_df.customer_id.nunique():,} users.')
print()
print('Label Distribution:')
print(val_df.churn.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
print(val_df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print(f'Customers in val seen in train: {len(set(val_df.customer_id) & set(train_df.customer_id)):,}')
print()
print('Temporal dist of users (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    val_df
    .drop_duplicates(subset='customer_id', keep='last')
    .groupby('timestamp').customer_id.nunique() / val_df.customer_id.nunique()
)

### `rel-amazon-item-ltv`

Task is to predict the revenue generated by item sales in the next time window.

Regression

Metric: MAE

In [ ]:
%%sql train_df <<
from item_ltv_train;

In [ ]:
train_df.head()

In [ ]:
print('TRAIN\n')

print(f'{len(train_df):,} samples over {train_df.product_id.nunique():,} products.')
print()
print('Temporal dist of samples:')
print(train_df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print('Temporal dist of products (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    train_df
    .drop_duplicates(subset='product_id', keep='last')
    .groupby('timestamp').product_id.nunique() / train_df.product_id.nunique()
)
print()
print('Label Distribution:')
mean = train_df.ltv.mean()
std = train_df.ltv.std()
print(f'Mean (std): {mean:,.1f} ± {std:,.1f}')
print(f'Prop zeros: {(train_df.ltv == 0).mean():.1%}')
print(f'Quantiles: \n{train_df.ltv.quantile(np.arange(0, 1.1, 0.1))}')

In [ ]:
%%sql val_df <<
from item_ltv_val;

In [ ]:
print('VAL\n')

print(f'{len(val_df):,} samples over {val_df.product_id.nunique():,} products.')
print()
print('Temporal dist of samples:')
print(val_df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print('Temporal dist of products (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    val_df
    .drop_duplicates(subset='product_id', keep='last')
    .groupby('timestamp').product_id.nunique() / val_df.product_id.nunique()
)
print()
print(f'Products in val seen in train: {len(set(val_df.product_id) & set(train_df.product_id)):,}')
print()
print('Label Distribution:')
mean = val_df.ltv.mean()
std = val_df.ltv.std()
print(f'Mean (std): {mean:,.1f} ± {std:,.1f}')
print(f'Prop zeros: {(val_df.ltv == 0).mean():.1%}')
print(f'Quantiles: \n{val_df.ltv.quantile(np.arange(0, 1.1, 0.1))}')

### `rel-amazon-item-churn`

Task is to predict whether item is not going to sell in the next time window

Binary Classification

Metric: ROCAUC

In [ ]:
%%sql train_df <<
from item_churn_train;

In [ ]:
print('TRAIN\n')

print(f'{len(train_df):,} samples over {train_df.product_id.nunique():,} users.')
print()
print('Label Distribution:')
print(train_df.churn.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
print(train_df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print('Temporal dist of users (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    train_df
    .drop_duplicates(subset='product_id', keep='last')
    .groupby('timestamp').product_id.nunique() / train_df.product_id.nunique()
)

In [ ]:
%%sql val_df <<
from item_churn_val;

In [ ]:
print('VAL\n')

print(f'{len(val_df):,} samples over {val_df.product_id.nunique():,} users.')
print()
print('Label Distribution:')
print(val_df.churn.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
print(val_df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print(f'Products in val seen in train: {len(set(val_df.product_id) & set(train_df.product_id)):,}')
print()
print('Temporal dist of users (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    val_df
    .drop_duplicates(subset='product_id', keep='last')
    .groupby('timestamp').product_id.nunique() / val_df.product_id.nunique()
)

## Customer

In [ ]:
%%sql customer << 
from customer;

In [ ]:
customer.head()

In [ ]:
customer['name_len'] = customer.customer_name.str.len()

In [ ]:
customer.name_len.quantile(np.arange(0, 1.1, 0.1))

In [ ]:
customer[customer.name_len > 50]

## Product

In [ ]:
%%sql product <<
from product;

In [ ]:
product.head()

In [ ]:
fig = px.histogram(x=product.price, range_x=(0, 100))
fig.update_layout(width=700, height=500, xaxis_title='Price', yaxis_title='Count')
fig.show(renderer='png')

In [ ]:
product.category.iloc[0]

In [ ]:
def get_specific(x):
    if isinstance(x, float):
        return ' '
    return str(x[-1])

cats = product.category.apply(get_specific)

In [ ]:
cats.value_counts()

In [ ]:
product['desc_len'] = product.description.str.len()
product['title_len'] = product.title.str.len()

In [ ]:
product.desc_len.quantile(np.arange(0, 1.1, 0.1))

In [ ]:
product.title_len.quantile(np.arange(0, 1.1, 0.1))

## Review

In [ ]:
%%sql review <<
-- review is a large table beware of memory usage
from review;

In [ ]:
review.head()

In [ ]:
review.verified.value_counts()

In [ ]:
review.groupby('verified').rating.mean()

In [ ]:
%%sql per_product <<
select
    product_id,
    avg(rating) as avg_rating,
    count(*) as num_reviews
from review
group by product_id

In [ ]:
fig = px.histogram(per_product, x='avg_rating', nbins=50)
fig.update_layout(width=700, height=500)
fig.show(renderer='png')

In [ ]:
fig = px.histogram(per_product, x='num_reviews', nbins=50, log_y=True)
fig.update_layout(width=700, height=500)
fig.show(renderer='png')

In [ ]:
%%sql per_customer <<
select
    customer_id,
    avg(rating) as avg_rating,
    count(*) as num_reviews
from review
group by customer_id

In [ ]:
fig = px.histogram(per_customer, x='avg_rating', nbins=50)
fig.update_layout(width=700, height=500)
fig.show(renderer='png')

In [ ]:
fig = px.histogram(per_customer, x='num_reviews', nbins=50, log_y=True)
fig.update_layout(width=700, height=500)
fig.show(renderer='png')

In [ ]:
%%sql weeks_bw_reviews <<
select
    date_diff(
        'weeks',
        review_time,
        lead(review_time, 1) over (partition by customer_id order by review_time)
    ) as weeks_bw_reviews
from review

In [ ]:
fig = px.histogram(weeks_bw_reviews, x='weeks_bw_reviews', log_y=True)
fig.update_layout(width=700, height=500)
fig.show(renderer='png')

In [ ]:
weeks_bw_reviews.weeks_bw_reviews.quantile(np.arange(0, 1.1, 0.1))

In [ ]:
weeks_bw_reviews.weeks_bw_reviews.isna().mean()